In [ ]:
import pandas as pd
from transformers import AutoTokenizer
import torch

# Load data
df = pd.read_csv('/content/train.csv')  # adjust path as needed

# ---------------------------------------------------------
# Q1: Label Encoding
# ---------------------------------------------------------
label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
df['label'] = df['answer'].map(label_map)

print("Q1 - Encoded label at index 150:", df.loc[150, 'label'])


Q1 - Encoded label at index 150: 2


In [ ]:
# ---------------------------------------------------------
# Q2: Prompt-Option Formatting
# ---------------------------------------------------------
prompt = df.loc[0, 'prompt']
option_B = df.loc[0, 'B']

formatted_B = str(prompt) + " [SEP] " + str(option_B)

print("Q2 - Formatted input length:", len(formatted_B))

Q2 - Formatted input length: 407


In [ ]:

# ---------------------------------------------------------
# Q3: Single-Row MCQ Tokenization
# ---------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

options = ['A', 'B', 'C', 'D', 'E']
row = df.loc[0]

# Build 5 formatted (prompt, option) pairs for this single row
first_sentences = [str(row['prompt'])] * 5
second_sentences = [str(row[opt]) for opt in options]

encoding = tokenizer(
    first_sentences,
    second_sentences,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Reshape to [batch_size, num_choices, seq_len]
input_ids = encoding['input_ids'].view(1, 5, 128)

print("Q3 - input_ids shape:", input_ids.shape)
print("Q3 - Second dimension (num_choices):", input_ids.shape[1])

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Q3 - input_ids shape: torch.Size([1, 5, 128])
Q3 - Second dimension (num_choices): 5


In [ ]:
# ---------------------------------------------------------
# Q4: Batch MCQ Tokenization (first 16 rows)
# ---------------------------------------------------------
batch_df = df.iloc[:16]

first_sentences = []
second_sentences = []

for _, r in batch_df.iterrows():
    first_sentences.extend([str(r['prompt'])] * 5)
    second_sentences.extend([str(r[opt]) for opt in options])

batch_encoding = tokenizer(
    first_sentences,
    second_sentences,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Reshape to [16, 5, 128]
batch_input_ids = batch_encoding['input_ids'].view(16, 5, 128)

print("Q4 - input_ids shape:", batch_input_ids.shape)
print("Q4 - Total token positions:", batch_input_ids.numel())

Q4 - input_ids shape: torch.Size([16, 5, 128])
Q4 - Total token positions: 10240


In [ ]:
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType

# ---------------------------------------------------------
# Setup
# ---------------------------------------------------------
df = pd.read_csv('/content/train.csv')  # adjust path as needed

label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
df['label'] = df['answer'].map(label_map)

options = ['A', 'B', 'C', 'D', 'E']
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)


def tokenize_row(row, max_length=128):
    """Tokenize a single row into 5 (prompt, option) pairs -> [5, max_length]"""
    first_sentences = [str(row['prompt'])] * 5
    second_sentences = [str(row[opt]) for opt in options]

    encoding = tokenizer(
        first_sentences,
        second_sentences,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return encoding


In [ ]:
# ---------------------------------------------------------
# Q5: Multiple-Choice Logits
# ---------------------------------------------------------
model = AutoModelForMultipleChoice.from_pretrained(model_name)

row0 = df.loc[0]
encoding0 = tokenize_row(row0, max_length=128)

input_ids = encoding0['input_ids'].unsqueeze(0)          # [1, 5, 128]
attention_mask = encoding0['attention_mask'].unsqueeze(0)  # [1, 5, 128]

with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)

logits = outputs.logits
print("Q5 - logits shape:", logits.shape)
print("Q5 - number of logits for one question:", logits.shape[1])


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Q5 - logits shape: torch.Size([1, 5])
Q5 - number of logits for one question: 5


In [ ]:


# ---------------------------------------------------------
# Q6: Supervised Loss Tensor
# ---------------------------------------------------------
label0 = torch.tensor([row0['label']])  # correct encoded label for row 0

with torch.no_grad():
    outputs_with_loss = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=label0
    )

loss = outputs_with_loss.loss
print("Q6 - loss value:", loss.item())
print("Q6 - loss tensor ndim:", loss.dim())  # dimensions of the loss tensor

Q6 - loss value: 1.6186108589172363
Q6 - loss tensor ndim: 0


In [ ]:
import sys
!{sys.executable} -m pip install --upgrade torchao

# ---------------------------------------------------------
# Q7: LoRA Trainable Parameters
# ---------------------------------------------------------
base_model = AutoModelForMultipleChoice.from_pretrained(model_name)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)
lora_model = get_peft_model(base_model, lora_config)

trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
print("Q7 - Trainable parameters:", trainable_params)

lora_model.print_trainable_parameters()  # optional, human-readable summary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.5 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Q7 - Trainable parameters: 295681
trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693


In [ ]:
!pip install "datasets==2.19.0" "torchvision==0.17.2" "torch==2.2.2" --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
import sys
# ---------------------------------------------------------
# Q8: Hugging Face Dataset Preparation
# ---------------------------------------------------------
def build_examples(dataframe, max_length=128):
    examples = {
        "input_ids": [],
        "attention_mask": [],
        "labels": []
    }
    for _, row in dataframe.iterrows():
        enc = tokenize_row(row, max_length=max_length)
        examples["input_ids"].append(enc["input_ids"])           # [5, max_length]
        examples["attention_mask"].append(enc["attention_mask"]) # [5, max_length]
        examples["labels"].append(row["label"])
    return examples


subset_100 = df.iloc[:100].reset_index(drop=True)
data_dict = build_examples(subset_100, max_length=128)

hf_dataset = Dataset.from_dict({
    "input_ids": [x.tolist() for x in data_dict["input_ids"]],
    "attention_mask": [x.tolist() for x in data_dict["attention_mask"]],
    "labels": data_dict["labels"]
})

hf_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

first_item = hf_dataset[0]
print("Q8 - input_ids shape for first item:", first_item["input_ids"].shape)
print("Q8 - number of tokenized choices:", first_item["input_ids"].shape[0])

ImportError: cannot import name 'VideoReader' from 'torchvision.io' (/usr/local/lib/python3.12/dist-packages/torchvision/io/__init__.py)

In [ ]:
# ---------------------------------------------------------
# Q9: Tiny LoRA Fine-Tuning
# ---------------------------------------------------------
subset_32 = df.iloc[:32].reset_index(drop=True)
train_data_dict = build_examples(subset_32, max_length=64)

train_dataset = Dataset.from_dict({
    "input_ids": [x.tolist() for x in train_data_dict["input_ids"]],
    "attention_mask": [x.tolist() for x in train_data_dict["attention_mask"]],
    "labels": train_data_dict["labels"]
})
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# Fresh LoRA model for fine-tuning
ft_base_model = AutoModelForMultipleChoice.from_pretrained(model_name)
ft_lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)
ft_model = get_peft_model(ft_base_model, ft_lora_config)

training_args = TrainingArguments(
    output_dir="./mcq_lora_output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=train_dataset
)

train_result = trainer.train()

print("Q9 - Final global_step:", trainer.state.global_step)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist

ImportError: cannot import name 'VideoReader' from 'torchvision.io' (/usr/local/lib/python3.12/dist-packages/torchvision/io/__init__.py)

In [ ]:
`# ---------------------------------------------------------
# Q10: Probability Assigned to Option E After Fine-Tuning
# ---------------------------------------------------------
ft_model.eval()

row0_ft = df.loc[0]
encoding0_ft = tokenize_row(row0_ft, max_length=64)  # match training max_length

input_ids_ft = encoding0_ft['input_ids'].unsqueeze(0)
attention_mask_ft = encoding0_ft['attention_mask'].unsqueeze(0)

with torch.no_grad():
    ft_outputs = ft_model(input_ids=input_ids_ft, attention_mask=attention_mask_ft)

ft_logits = ft_outputs.logits  # [1, 5]
probs = F.softmax(ft_logits, dim=-1)

prob_E = probs[0][4].item()  # index 4 corresponds to Option E
print("Q10 - Probability assigned to Option E:", round(prob_E, 4))

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Q10 - Probability assigned to Option E: 0.1853
